# Phase 3: Reconstruction Interpretability (25 Marks)

**Team Astra** | NSSC 2026 | IIT Kharagpur  
**Lead:** 

---

## Objectives
1. Select the **top 5 most anomalous images** that strictly exceed the threshold
2. Compute **per-pixel absolute error** between original and reconstruction
3. Render as **overlay heatmaps** (3-panel: Original | Reconstruction | Error)
4. Provide a **geological hypothesis** for each flagged image

### Physical Interpretation Guidelines
- Sharp linear features → spliced boundary artifact
- Diffuse errors → domain shift or sensor calibration change
- Periodic stripes → sensor readout / hardware glitch
- Localized bright spots → terrestrial interference injection
- Uniform elevated error → contrast/exposure anomaly

In [ ]:
# ─── System Setup ─────────────────────────────────────────────────────
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

IMAGE_DIR = os.path.join(PROJECT_ROOT, 'data', 'images')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'models')
LATENT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'latent_vectors')
PLOTS_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'plots')
SCORES_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'scores')
REPORT_DIR = os.path.join(PROJECT_ROOT, 'outputs', 'report')

In [ ]:
# ─── Imports ──────────────────────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import set_seed, get_device
from src.dataset import create_dataloaders
from src.models import get_model
from src.latent_utils import load_latents
from src.heatmaps import (
    top_k_anomaly_panels,
    generate_geological_report,
)

set_seed(42)
device = get_device()
print('Phase 3 imports ready.')

## 3.1 Load Best Model & Scores

In [ ]:
BEST_VERSION = 'v5'

# Load the best model checkpoint
model = get_model(BEST_VERSION)
checkpoint = torch.load(
    os.path.join(MODELS_DIR, f'best_{BEST_VERSION}.pth'),
    map_location=device,
    weights_only=False,
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()
print(f'Model {BEST_VERSION.upper()} loaded (epoch {checkpoint["epoch"]})')

# Load scores
scores_df = pd.read_csv(
    os.path.join(SCORES_DIR, f'novelty_scores_{BEST_VERSION}.csv')
)
print(f'Scores loaded: {len(scores_df)} images')
print(f'Anomalies flagged: {scores_df["is_anomaly"].sum()}')

In [ ]:
# Create full data loader
_, _, full_loader = create_dataloaders(
    image_dir=IMAGE_DIR,
    batch_size=32,
    val_split=0.1,
    num_workers=4,
)

# Get scores and filenames in dataset order
latents, filenames = load_latents(save_dir=LATENT_DIR, version=BEST_VERSION)

# Map filenames to scores
score_map = dict(zip(scores_df['filename'], scores_df['novelty_score']))
scores = np.array([score_map.get(fn, 0.0) for fn in filenames])

# Get effective threshold from consensus
anomaly_mask = scores_df[scores_df['is_anomaly'] == True]
if len(anomaly_mask) > 0:
    effective_threshold = anomaly_mask['novelty_score'].min()
else:
    effective_threshold = np.percentile(scores, 99)

print(f'Effective threshold: {effective_threshold:.6f}')
print(f'Images exceeding threshold: {(scores > effective_threshold).sum()}')

## 3.2 Generate Top-5 Anomaly Heatmaps

For each of the top-5 most anomalous images (strictly exceeding the threshold), we:
1. Pass through the autoencoder to get the reconstruction
2. Compute per-pixel absolute error: $E(i,j) = |X(i,j) - \hat{X}(i,j)|$
3. Render as a 3-panel figure: **Original** | **Reconstruction** | **Error Heatmap**

In [ ]:
# Generate heatmap panels for top-5 anomalies
anomaly_results = top_k_anomaly_panels(
    model=model,
    loader=full_loader,
    device=device,
    scores=scores,
    filenames=filenames,
    threshold=effective_threshold,
    k=5,
    save_dir=PLOTS_DIR,
    version=BEST_VERSION,
)

print(f'\nGenerated heatmaps for {len(anomaly_results)} anomalies')

## 3.3 Geological Hypothesis Report

In [ ]:
# Display results table
results_df = pd.DataFrame(anomaly_results)
print('\n' + '=' * 80)
print('  GEOLOGICAL HYPOTHESIS REPORT — TOP 5 ANOMALIES')
print('=' * 80)

for r in anomaly_results:
    print(f'\n  Anomaly #{r["rank"]} — {r["filename"]}')
    print(f'  Novelty Score: {r["novelty_score"]:.4f}')
    print(f'  Mean Error:    {r["mean_error"]:.4f}')
    print(f'  Max Error:     {r["max_error"]:.4f}')
    print(f'  Error Pattern: {r["error_pattern"]}')
    print(f'  Hypothesis:    {r["physical_hypothesis"]}')
    print('  ' + '─' * 76)

In [ ]:
# Generate markdown report
report = generate_geological_report(
    anomaly_results=anomaly_results,
    save_path=os.path.join(REPORT_DIR, 'geological_report.md'),
)

print('\n✓ Phase 3 complete. Heatmaps and geological report saved.')

## 3.4 Additional Analysis: Error Distribution Across Dataset

In [ ]:
# Compute mean reconstruction error per image for the entire dataset
from src.heatmaps import compute_reconstruction_errors

originals, errors, all_fnames = compute_reconstruction_errors(
    model, full_loader, device
)

mean_errors = errors.mean(axis=(1, 2))  # Per-image mean error

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Histogram of per-image mean errors
ax1.hist(mean_errors, bins=100, color='#3498db', alpha=0.7, edgecolor='white')
ax1.axvline(np.percentile(mean_errors, 99), color='red', linestyle='--',
            linewidth=2, label=f'99th percentile: {np.percentile(mean_errors, 99):.4f}')
ax1.set_xlabel('Mean Reconstruction Error', fontsize=12)
ax1.set_ylabel('Count', fontsize=12)
ax1.set_title('Per-Image Mean Reconstruction Error', fontsize=13, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Scatter: Novelty Score vs Mean Error
fname_to_error = dict(zip(all_fnames, mean_errors))
aligned_errors = np.array([fname_to_error.get(fn, 0) for fn in filenames])

ax2.scatter(scores, aligned_errors, s=3, alpha=0.4, c='#3498db')
ax2.set_xlabel('Novelty Score (Isolation Forest)', fontsize=12)
ax2.set_ylabel('Mean Reconstruction Error', fontsize=12)
ax2.set_title('Novelty Score vs. Reconstruction Error', fontsize=13, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'error_analysis.png'), dpi=150)
plt.show()